<h1 align="center">
<h1 align="center"> Tarea 2 - Web Structure Mining
<h2 align="center">  Integrantes: Cristobal Gajardo y Fernando Cáceres.
<h3 align="center"> IN5529 - Inteligencia Artificial Aplicada a la Web
<h3 align="center"> Primavera 2025

---

# Introducción

Este notebook implementa un prototipo simple de buscador (“Googlito”) sobre un sitio tipo wiki en este caso lo hicimos sobre la Wiki de Terraria en español (https://terraria.fandom.com/es). Este dominio es distinto al utilizado en la auxiliar y todas las páginas recolectadas provienen de este sitio base.
Se recorre el sitio con un web scraper para obtener texto y enlaces; se limpia el corpus y se construye una representación TF‑IDF con unigramas y bigramas.
Luego se implementan desde cero los algoritmos PageRank, HITS y SALSA para obtener rankings estructurales del grafo.
Finalmente, se definen funciones de búsqueda que usan esos rankings (y el contenido textual) y se realiza una demostración con consultas de una palabra y oraciones, seguida de una discusión de resultados.




#Instalación e importación de librerias

**Limpieza y normalización de texto**

Limpia el texto: minúsculas, eliminación de signos y stopwords, y aplica lematización o stemming para normalizar palabras antes de vectorizar.

In [ ]:
# Instalaciones
!pip install selenium
!apt-get update
!apt install chromium-chromedriver
import sys
sys.path.insert(0,'/usr/lib/chromium-browser/chromedriver')
!python -m spacy download es_core_news_md
!pip install nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.7/512.7 kB 31.3 MB/s eta 0:00:00
Hit:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://cli.github.com/packages stable/main amd64 Packages [344 B]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,816 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:11 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:1

**Representación TF‑IDF**

Construye la matriz TF‑IDF (unigramas y bigramas) a partir del texto limpio. Devuelve el vectorizador entrenado, la matriz dispersa y el vocabulario de términos.

In [ ]:
# Web Scraper
from selenium import webdriver
from selenium.common.exceptions import TimeoutException, StaleElementReferenceException
from selenium.common.exceptions import TimeoutException, WebDriverException
from selenium.webdriver.common.by import By




# Librerías varias
import pandas as pd
import numpy as np
import json
import time
import random
import urllib

# Preprocesamiento
import re
import unicodedata
import nltk
nltk.download('punkt')
nltk.download('stopwords')
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import strip_accents_ascii
import spacy
import string

# Ranking
from sklearn.feature_extraction.text import TfidfVectorizer
from collections import Counter
import networkx as nx

import nltk

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


# Obtención de datos

## Web Scraper

**Importaciones y configuración**

Carga librerías y configura constantes, rutas y parámetros usados en el notebook.

In [ ]:
# Configuración del Webdriver
from selenium.webdriver.chrome.service import Service
chrome_options = webdriver.ChromeOptions()
chrome_options.add_argument("--headless=new")
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")
chrome_options.add_argument("--lang=es-ES")
chrome_options.add_argument("--user-agent=Mozilla/5.0")

prefs = {
    "profile.managed_default_content_settings.images": 2,
    "profile.managed_default_content_settings.stylesheets": 2,
    "profile.managed_default_content_settings.cookies": 1
}
chrome_options.add_experimental_option("prefs", prefs)

caps = chrome_options.to_capabilities()
caps["pageLoadStrategy"] = "eager"

service = Service()


**Normalización de URLs**

Estandariza los enlaces para evitar duplicados por variaciones (slash final, anclas, parámetros). Se usa como paso previo al crawling y a la construcción del grafo.

In [ ]:
# Función get_links (modificada para buscar en contenido principal)
from urllib.parse import urljoin, urlparse, urlunparse  # Para manejar URLs
from selenium import webdriver
from selenium.webdriver.common.by import By
from urllib.parse import urlparse, urlunparse
from selenium.common.exceptions import TimeoutException, StaleElementReferenceException # Importar TimeoutException

# Normalizar la URL para evitar duplicados
def normalize_url(url):
    p = urlparse(url)._replace(query="", fragment="")
    return urlunparse(p).rstrip("/")  # Eliminar fragmentos y parámetros innecesarios

def get_links(url, url_base, url_data, seen):
    global chrome_options, service
    total = 0
    contador_visitas = 0  # Contador para las primeras 100 visitas
    max_mostrar = 100  # Mostrar solo las primeras 100 páginas

    driver = None # Inicializar driver a None
    try:
        driver = webdriver.Chrome(service=service, options=chrome_options)
        # Establecer un tiempo de espera para la carga de la página
        driver.set_page_load_timeout(30) # Puedes ajustar este valor según necesites

        driver.get(url)

        links = []
        # 1) Intento: buscar enlaces dentro de mw-parser-output
        try:
            content_div = driver.find_element(By.CSS_SELECTOR, 'div.mw-parser-output')
            links = content_div.find_elements(By.TAG_NAME, "a")
        except Exception:
            pass # Si no se encuentra, intentar el siguiente selector

        # 2) Fallback: #mw-content-text
        if not links:
            try:
                content_div = driver.find_element(By.CSS_SELECTOR, '#mw-content-text')
                links = content_div.find_elements(By.TAG_NAME, "a")
            except Exception:
                pass # Si no se encuentra, intentar el último recurso

        # 3) Último recurso: buscar en todo el cuerpo si no se encontraron enlaces en las secciones principales
        if not links:
             try:
                 links = driver.find_elements(By.TAG_NAME, "a")
             except Exception:
                 pass


        for link in links:
            href = link.get_attribute('href')
            if href:
                normalized_url = normalize_url(href)  # Normalizar la URL

                # Filtrar enlaces irrelevantes o externos
                if (url_base not in normalized_url) or any(x in normalized_url.lower() for x in ["discord", "twitter", "facebook", "youtube"]):
                    continue  # Ignorar enlaces externos

                # Filtrar enlaces que no son útiles (por ejemplo, "Wiki_Terraria")
                # Mantener este filtro para evitar rastrear estas paginas en url_data
                # Para urls_out, si estas paginas estan en url_data, sus enlaces salientes seran considerados
                if "wiki_terraria" in normalized_url.lower() or "guia_del_principiante" in normalized_url.lower():
                    continue  # Ignorar páginas como "Wiki_Terraria" o la "Guía del principiante"


                if normalized_url not in seen:
                    # Agregar el enlace a la lista si es relevante y no ha sido visto
                    # Esta logica de 'seen' es mas critica para la recoleccion inicial de url_data
                    # Para urls_out, 'seen' ayuda a evitar duplicados dentro de la misma pagina
                    url_data.append(normalized_url)
                    seen.add(normalized_url)
                    total += 1
                    contador_visitas += 1

                    # Mostrar las primeras 100 páginas visitadas
                    if contador_visitas <= max_mostrar:
                        print(f"Visitando: {normalized_url} ({contador_visitas}/{max_mostrar})")

        print(f"Total de enlaces encontrados en esta página: {total}")

    except TimeoutException:
        print(f"\nTiempo de carga excedido para: {url}. Saltando.")
    except Exception as e:
        print(f"\nError al obtener enlaces para {url}: {e}")
    finally:
        if driver: # Asegurarse de que driver fue inicializado antes de intentar cerrarlo
            try:
                driver.quit()
            except:
                pass

**Web Scraper: extracción de páginas**

Descarga páginas del sitio https://terraria.fandom.com/es , extrae el texto principal y los enlaces salientes (href). Devuelve estructuras listas para construir el grafo y el dataset.

El crawler sigue enlaces internos de este dominio, normaliza URLs y descarta enlaces externos.

In [ ]:
# Web Scraper
def get_text(url):
    d = webdriver.Chrome(service=service, options=chrome_options)
    try:
        # timeouts cortos para evitar bloqueos
        try:
            d.set_page_load_timeout(25)
            d.set_script_timeout(8)
        except Exception:
            pass

        try:
            d.get(url)
        except TimeoutException:
            # si tarda demasiado cargando, seguimos con lo que haya
            pass

        # 1) Intento: mw-parser-output
        try:
            js1 = "var el=document.querySelector('div.mw-parser-output'); return el?el.innerText:'';"
            txt = d.execute_script(js1) or ""
            txt = (txt or "").strip()
            if txt:
                return [txt]
        except WebDriverException:
            pass

        # 2) Fallback: #mw-content-text
        try:
            js2 = "var el=document.querySelector('#mw-content-text'); return el?el.innerText:'';"
            txt2 = d.execute_script(js2) or ""
            txt2 = (txt2 or "").strip()
            if txt2:
                return [txt2]
        except WebDriverException:
            pass

        # 3) Último recurso: body
        try:
            js3 = "return (document.body && document.body.innerText) ? document.body.innerText : '';"
            body_txt = d.execute_script(js3) or ""
            return [ (body_txt or '').strip() ]
        except WebDriverException:
            return [""]

    except Exception as e:
        print("get_text error:", e)
        return [""]
    finally:
        try:
            d.quit()
        except:
            pass

In [ ]:
# Scrapeamos las URLs desde la página principal
url_base = 'https://terraria.fandom.com/es'  # terraria :)
url_data = [url_base]  # Lista de se almacenan las URLs encontradas
seen_urls = {url_base} # Conjunto para URLs ya vistas
n_urls = 50  # Cantidad de URLs que queremos recolectar (reducido para prueba)

# Usamos un índice para iterar y poder añadir nuevos elementos a url_data
i = 0
while i < len(url_data) and len(seen_urls) < n_urls:
    url = url_data[i]
    # Limitar la impresion para no saturar la salida si n_urls es grande
    if i < 10 or i % 10 == 0:
        print(f"Procesando: {url} ({i+1}/{len(url_data)}) - Vistas: {len(seen_urls)}")
    get_links(url, url_base, url_data, seen_urls)
    i += 1

# Nos aseguramos de tener exactamente n_urls únicas (si es posible)
# Si recolectamos mas de n_urls en el proceso, tomamos una muestra aleatoria
if len(seen_urls) > n_urls:
  url_data = random.sample(list(seen_urls), n_urls)
else:
  url_data = list(seen_urls)


print('Cantidad final de urls únicas:', len(url_data))

Procesando: https://terraria.fandom.com/es (1/1) - Vistas: 1
Visitando: https://terraria.fandom.com/es/wiki/Gu%C3%ADa_del_principiante (1/100)
Visitando: https://terraria.fandom.com/es/wiki/Terraria (2/100)
Visitando: https://terraria.fandom.com/es/wiki/Re-Logic (3/100)
Visitando: https://terraria.fandom.com/es/wiki/Clases (4/100)
Visitando: https://terraria.fandom.com/es/wiki/1.4.4.9 (5/100)
Visitando: https://terraria.fandom.com/es/wiki/Terraria_en_consolas (6/100)
Visitando: https://terraria.fandom.com/es/wiki/1.2.4.1 (7/100)
Visitando: https://terraria.fandom.com/es/wiki/1.2.1 (8/100)
Visitando: https://terraria.fandom.com/es/wiki/Mod:TModLoader (9/100)
Visitando: https://terraria.fandom.com/es/wiki/Fulgor (10/100)
Visitando: https://terraria.fandom.com/es/wiki/%C3%89ter (11/100)
Visitando: https://terraria.fandom.com/es/wiki/Modos_de_juego (12/100)
Visitando: https://terraria.fandom.com/es/wiki/Mods (13/100)
Visitando: https://terraria.fandom.com/es/wiki/Pesca (14/100)
Visitando: 

In [ ]:
# Scrapeamos el texto de cada url (con conteo OK/saltados)
url_text = []
i = 0
ok = 0
skipped = 0

for url in url_data:
  print(f"\rLinks scrapeados: {i}/{len(url_data)}", end="")
  try:
      t = get_text(url)[0]
      if t:
          url_text.append(t)
          ok += 1
      else:
          skipped += 1
  except KeyboardInterrupt:
      raise
  except Exception:
      skipped += 1
  i += 1

print(f"\rLinks scrapeados: {i}/{len(url_data)}")
print(f"Leídos OK: {ok} | Saltados: {skipped} | Total: {len(url_data)}")

url_text


Links scrapeados: 1/50get_text error: HTTPConnectionPool(host='localhost', port=58767): Read timed out. (read timeout=120)


Links scrapeados: 50/50
Leídos OK: 49 | Saltados: 1 | Total: 50


['Armadura de las sombras\nMasculino\nFemenino\nOtros nombres\nInglés\nShadow armor\nCaracterísticas\nTipo\nArmadura\nRareza\nNivel 1 (Azul)\nDefensa\n19\nBonificación\nAumenta la velocidad de movimiento y aceleración.\nVenta\n1  80 \nCreación\nCreado con\n60 Lingote endemoniado\n45 Escama de las sombras\nCreado en\nYunque de hierro\nYunque de plomo\n\nTodos los efectos de la Armadura de las sombras, incluidas las llamas moradas de las Botas cohete.\n\nLa Armadura de las sombras (Shadow armor en inglés) es una armadura del modo normal que solamente se puede conseguir en mundos con corrupción ya que es creada con lingotes endemoniados y escamas de las sombras.\n\nEn sin versión, cada una de las 3 piezas proporciona un +5% otorga probabilidad de golpe crítico universal (para un total de +15%), mientras que llevar las 3 piezas a la vez otorga una bonificación adicional de +15% a la velocidad de movimiento y aceleración.\n\nEn sin versión, cada una de las 3 piezas proporciona un +7% a la v

In [ ]:
print(i,ok,skipped)

50 49 1


**Web Scraper: extracción de páginas**

Descarga páginas del sitio tipo wiki, extrae el texto principal y los enlaces salientes (href). Devuelve estructuras listas para construir el grafo y el dataset.

In [ ]:
# Scrapeamos los links de salida de cada url
urls_out = []
ok = 0
skipped = 0
total = len(url_data)

# Creamos un conjunto seen_urls_out para esta parte, para evitar duplicados en los links de salida si es necesario
# Aunque para la construccion del grafo, no es estrictamente necesario que los links de salida sean unicos
# en esta lista, si lo es para el conjunto global de urls recolectadas.
# Sin embargo, para mantener la consistencia y evitar posibles problemas, vamos a usar un conjunto aqui tambien.
seen_urls_out = set() # Conjunto para URLs ya vistas en esta etapa

for i, url in enumerate(url_data, 1):
    links_page = []
    print(f"({i}/{total})", end=" ")
    try:
        # Pasamos el conjunto seen_urls_out a la funcion get_links
        # Aunque get_links lo usara para evitar duplicados dentro de la misma llamada,
        # aqui lo usamos principalmente para mantener la firma de la funcion consistente.
        # La logica de agregar a links_page y no a un conjunto global es intencional para
        # mantener la lista de links de salida por cada pagina.
        get_links(url, url_base, links_page, seen_urls_out)
        urls_out.append(links_page)
        ok += 1
    except KeyboardInterrupt:
        raise
    except Exception as e:
        print("\nSaltado por error:", e)
        urls_out.append([])
        skipped += 1

print(f"\nPáginas procesadas OK: {ok} | Páginas saltadas: {skipped} | Total: {total}")

(1/50) Visitando: https://terraria.fandom.com/es/wiki/Rareza (1/100)
Visitando: https://terraria.fandom.com/es/wiki/Defensa (2/100)
Visitando: https://terraria.fandom.com/es/wiki/Lingote_endemoniado (3/100)
Visitando: https://terraria.fandom.com/es/wiki/Escama_de_las_sombras (4/100)
Visitando: https://terraria.fandom.com/es/wiki/Yunque_de_hierro (5/100)
Visitando: https://terraria.fandom.com/es/wiki/Yunque_de_plomo (6/100)
Visitando: https://terraria.fandom.com/es/wiki/Armaduras (7/100)
Visitando: https://terraria.fandom.com/es/wiki/Modo_normal (8/100)
Visitando: https://terraria.fandom.com/es/wiki/Corrupci%C3%B3n (9/100)
Visitando: https://terraria.fandom.com/es/wiki/Golpe_cr%C3%ADtico (10/100)
Visitando: https://terraria.fandom.com/es/wiki/Velocidad (11/100)
Visitando: https://terraria.fandom.com/es/wiki/Velocidad_de_ataque (12/100)
Visitando: https://terraria.fandom.com/es/wiki/Armadura_de_los_muertos (13/100)
Visitando: https://terraria.fandom.com/es/wiki/Botas_cohete (14/100)
Visi

**Construcción del dataset**

Arma el DataFrame con las columnas URL, texto y enlaces salientes. Sirve como insumo para TF‑IDF y los algoritmos de ranking.

In [ ]:
# Construimos un dataframe y lo guardamos
df = pd.DataFrame(list(zip(url_data, url_text, urls_out)), columns=['url', 'text', 'urls_out'])
df.to_pickle('wiki_terraria.pkl')
df.head()

,url,text,urls_out
0,https://terraria.fandom.com/es/wiki/Armadura_d...,Armadura de las sombras\nMasculino\nFemenino\n...,"[https://terraria.fandom.com/es/wiki/Rareza, h..."
1,https://terraria.fandom.com/es/wiki/Expriminator,Mineral de tungsteno\nCaracterísticas\nTipo\nB...,[https://terraria.fandom.com/es/wiki/Decoracio...
2,https://terraria.fandom.com/es/wiki/Mineral_de...,Las armas mágicas son un tipo de armas en Terr...,[https://terraria.fandom.com/es/wiki/Lingote_d...
3,https://terraria.fandom.com/es/wiki/Armas_m%C3...,Lingote de estaño\nOtros nombres\nInglés\nTin ...,[https://terraria.fandom.com/es/wiki/Man%C3%A1...
4,https://terraria.fandom.com/es/wiki/Lingote_de...,El inframundo o infierno es la capa más profun...,[https://terraria.fandom.com/es/wiki/Inframund...


## Tratamiento de datos

**Persistencia del dataset**

Guarda el DataFrame en formato `.pkl` para reutilizarlo sin ejecutar nuevamente el crawling.

In [ ]:
# Cargamos los datos
df = pd.read_pickle('wiki_terraria.pkl')
df.head()

,url,text,urls_out
0,https://terraria.fandom.com/es/wiki/Armadura_d...,Armadura de las sombras\nMasculino\nFemenino\n...,"[https://terraria.fandom.com/es/wiki/Rareza, h..."
1,https://terraria.fandom.com/es/wiki/Expriminator,Mineral de tungsteno\nCaracterísticas\nTipo\nB...,[https://terraria.fandom.com/es/wiki/Decoracio...
2,https://terraria.fandom.com/es/wiki/Mineral_de...,Las armas mágicas son un tipo de armas en Terr...,[https://terraria.fandom.com/es/wiki/Lingote_d...
3,https://terraria.fandom.com/es/wiki/Armas_m%C3...,Lingote de estaño\nOtros nombres\nInglés\nTin ...,[https://terraria.fandom.com/es/wiki/Man%C3%A1...
4,https://terraria.fandom.com/es/wiki/Lingote_de...,El inframundo o infierno es la capa más profun...,[https://terraria.fandom.com/es/wiki/Inframund...


In [33]:
from google.colab import files
files.download('wiki_terraria.pkl')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Limpieza y normalización de texto**

Limpia el texto: minúsculas, eliminación de signos y stopwords, y aplica lematización o stemming para normalizar palabras antes de vectorizar.

In [ ]:
# Función text_treatment
# Recibe un texto de entrada, retornandolo limpio en forma de lemmas
def text_treatment(text):

  #Eliminar tags html
  html_tags = re.compile('<.*?>|&([a-z0-9]+|#[0-9]{1,6}|#x[0-9a-f]{1,6});')
  text = re.sub(html_tags, ' ', text)

  #Eliminar acentos
  text = strip_accents_ascii(text)

  #Pasar a minúscula
  text = text.lower()

  #Eliminar puntuación
  regex = re.compile('[%s]' % re.escape(string.punctuation))
  text = regex.sub('', text)

  #Eliminar espacios adicionales
  text = ' '.join(text.split())

  #Remover stopwords
  spanish_stopwords = stopwords.words('spanish')
  text = ' '.join(word for word in text.split() if word not in spanish_stopwords)

  #Lematizacion
  nlp = spacy.load('es_core_news_md')
  text = nlp(text)
  text = ' '.join(tok.lemma_.lower() for tok in text)

  return text

In [ ]:
# Limpiamos los textos de cada url
df['text'] = df['text'].apply(lambda x: text_treatment(x))
df.head()

,url,text,urls_out
0,https://terraria.fandom.com/es/wiki/Armadura_d...,armadura sombra masculino femenino nombres ing...,"[https://terraria.fandom.com/es/wiki/Rareza, h..."
1,https://terraria.fandom.com/es/wiki/Expriminator,mineral tungsteno caracteristica tipo bloque s...,[https://terraria.fandom.com/es/wiki/Decoracio...
2,https://terraria.fandom.com/es/wiki/Mineral_de...,arma magica tipo arma terrario principal carac...,[https://terraria.fandom.com/es/wiki/Lingote_d...
3,https://terraria.fandom.com/es/wiki/Armas_m%C3...,lingote estano nombres ingles tin bar caracter...,[https://terraria.fandom.com/es/wiki/Man%C3%A1...
4,https://terraria.fandom.com/es/wiki/Lingote_de...,inframundo infierno capa mas profundo mundo co...,[https://terraria.fandom.com/es/wiki/Inframund...


# Ranking de páginas

## WordRank

**Palabras relevantes**:
itera sobre cada documento hasta dar con las 150 palabras mas relevantes.

In [ ]:
# Función get_top_n
# retorna una lista de las n palabras mas relevantes para cada documento

def get_top_n(terms, X, n=150):
    tops = []
    for i in range(X.shape[0]):
        row = X.getrow(i)
        if row.nnz == 0:
            tops.append({})
            continue

        idx = row.indices         # índices de términos presentes en el doc i
        vals = row.data           # pesos TF-IDF correspondientes

        # Si hay más de n términos en el doc, se toma el topn sin ordenar todo
        if len(vals) > n:
            topk_idx = np.argpartition(vals, -n)[-n:]
            # se ordena desc en los n seleccionados
            order = topk_idx[np.argsort(vals[topk_idx])[::-1]]
        else:
            order = np.argsort(vals)[::-1]

        feats = idx[order]
        weights = vals[order]     # pesos TF-IDF ordenados DESC
        # dict termino ---> peso (incluye unigramas y bigramas si es que el vectorizados los gener´p)
        tops.append({terms[j]: float(w) for j, w in zip(feats, weights)})

    return tops


**Representación TF‑IDF**

Construye la matriz TF‑IDF (unigramas y bigramas) a partir del texto limpio. Devuelve el vectorizador entrenado, la matriz dispersa y el vocabulario de términos.

In [ ]:
# - ngram_range=(1,2) (unigramas + bigramas).
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),   #unigramas y bigramas
    min_df=2,
    lowercase=False       # ya se hizo
)

# Matriz documento–término de TF-IDF
X = vectorizer.fit_transform(df['text'])

# Vocabulario alineado a X
terms = vectorizer.get_feature_names_out()

# Construcción de 'wordrank' por documento
df['wordrank'] = get_top_n(terms, X, n=50)




validaciones o visualizaciones menores del resultado.

In [ ]:
df.head()

,url,text,urls_out,wordrank
0,https://terraria.fandom.com/es/wiki/Armadura_d...,armadura sombra masculino femenino nombres ing...,"[https://terraria.fandom.com/es/wiki/Rareza, h...","{'sombra': 0.39472556736096837, 'armadura': 0...."
1,https://terraria.fandom.com/es/wiki/Expriminator,mineral tungsteno caracteristica tipo bloque s...,[https://terraria.fandom.com/es/wiki/Decoracio...,"{'tungsteno': 0.582740486110008, 'mineral': 0...."
2,https://terraria.fandom.com/es/wiki/Mineral_de...,arma magica tipo arma terrario principal carac...,[https://terraria.fandom.com/es/wiki/Lingote_d...,"{'ninguno': 0.46449852845207407, 'normal': 0.2..."
3,https://terraria.fandom.com/es/wiki/Armas_m%C3...,lingote estano nombres ingles tin bar caracter...,[https://terraria.fandom.com/es/wiki/Man%C3%A1...,"{'estano': 0.5095652926316093, 'lingote': 0.40..."
4,https://terraria.fandom.com/es/wiki/Lingote_de...,inframundo infierno capa mas profundo mundo co...,[https://terraria.fandom.com/es/wiki/Inframund...,"{'inframundo': 0.3051795970080333, 'ahora': 0...."


## PageRank

In [ ]:
pagerank = np.zeros((len(df), len(df)), dtype=np.float32)

# Mapa URL -> índice para acelerar la búsqueda de columnas
url_to_idx = {u: i for i, u in enumerate(df['url'])}

# Construimos la matriz de adyacencia
for i, row in df.iterrows():
  outs = row['urls_out'] or []
  for url in outs:
    j = url_to_idx.get(url, None)  # solo si la URL destino está en el corpus
    if j is not None:
      pagerank[i][j] = 1.0

# outdeg: grado de salida por fila (nodo)
outdeg = pagerank.sum(axis=1).astype(int)

# inlinks: lista de predecesores por columna (quiénes apuntan a j)
inlinks = [np.where(pagerank[:, j] != 0)[0].tolist() for j in range(len(df))]

pagerank


array([[1., 0., 0., ..., 0., 0., 0.],
       [0., 1., 1., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], dtype=float32)

**PageRank desde cero**

Implementa PageRank parametrizable. Se construye la matriz de transición a partir de los enlaces, se maneja el damping factor y nodos colgantes, y se itera hasta convergencia para obtener el puntaje de cada página.

In [ ]:
# Implementación de PageRank
def pagerank_from_scratch(inlinks, outdeg, d=0.85, tol=1e-6, max_iter=100):
    """
    inlinks[i]: lista de nodos j con enlace j -> i
    outdeg[j]:  grado de salida de j
    entrega: vector PR (suma = 1).
    """
    n = len(inlinks)
    pr = np.full(n, 1.0 / n, dtype=np.float64)
    base = (1.0 - d) / n

    for _ in range(max_iter):
        dangling_mass = pr[outdeg == 0].sum()

        # nuevo PR parte en el término base + contrib. de dangling
        new_pr = np.full(n, base, dtype=np.float64)
        new_pr += d * (dangling_mass / n)

        # aportes entrantes
        for i in range(n):
            s = 0.0
            for j in inlinks[i]:
                s += pr[j] / outdeg[j]
            new_pr[i] += d * s

        # convergencia
        if np.abs(new_pr - pr).sum() < tol:
            pr = new_pr
            break
        pr = new_pr

    pr /= pr.sum()
    return pr

# d=0.85
pr_085 = pagerank_from_scratch(inlinks, outdeg, d=0.85, tol=1e-6, max_iter=100)
pr_060 = pagerank_from_scratch(inlinks, outdeg, d=0.60, tol=1e-6, max_iter=100)

df['pr_085']  = pr_085
df['pr_060'] = pr_060
df.head()


,url,text,urls_out,wordrank,pr_085,pr_060
0,https://terraria.fandom.com/es/wiki/Armadura_d...,armadura sombra masculino femenino nombres ing...,"[https://terraria.fandom.com/es/wiki/Rareza, h...","{'sombra': 0.39472556736096837, 'armadura': 0....",0.010894,0.015736
1,https://terraria.fandom.com/es/wiki/Expriminator,mineral tungsteno caracteristica tipo bloque s...,[https://terraria.fandom.com/es/wiki/Decoracio...,"{'tungsteno': 0.582740486110008, 'mineral': 0....",0.011150,0.015985
2,https://terraria.fandom.com/es/wiki/Mineral_de...,arma magica tipo arma terrario principal carac...,[https://terraria.fandom.com/es/wiki/Lingote_d...,"{'ninguno': 0.46449852845207407, 'normal': 0.2...",0.011150,0.015985
3,https://terraria.fandom.com/es/wiki/Armas_m%C3...,lingote estano nombres ingles tin bar caracter...,[https://terraria.fandom.com/es/wiki/Man%C3%A1...,"{'estano': 0.5095652926316093, 'lingote': 0.40...",0.010709,0.015553
4,https://terraria.fandom.com/es/wiki/Lingote_de...,inframundo infierno capa mas profundo mundo co...,[https://terraria.fandom.com/es/wiki/Inframund...,"{'inframundo': 0.3051795970080333, 'ahora': 0....",0.012730,0.017584


## HITS
Este código implementa el algoritmo HITS (Hyperlink-Induced Topic Search), que calcula dos valores por página web en un grafo de enlaces:

Authorities = páginas que son buenas fuentes de información (las más enlazadas por buenos hubs).

Hubs = páginas que enlazan a buenas authorities
El código implementa el algoritmo HITS desde cero usando listas de adyacencia derivadas de la matriz pagerank.

Calcula iterativamente los valores de hub y authority hasta que convergen.

Devuelve los 10 principales hubs y las 10 principales authorities .

In [ ]:
# HITS

n = pagerank.shape[0]

# A quienes apunta i
outlinks_idx = [np.where(pagerank[i, :] != 0)[0].tolist() for i in range(n)]
# A quienes apunta j
inlinks_idx  = [np.where(pagerank[:, j] != 0)[0].tolist() for j in range(n)]

hubs = np.ones(n, dtype=np.float64)
auth = np.ones(n, dtype=np.float64)

# normalizacion
def _norm2(v):
    s = np.linalg.norm(v)
    return v / s if s > 0 else v

hubs = _norm2(hubs)
auth = _norm2(auth)
tol = 1e-6
max_iter = 200

for _ in range(max_iter):
    # Actualiza authorities
    new_auth = np.zeros(n, dtype=np.float64)
    for i in range(n):
        if inlinks_idx[i]:                        # si alguna pag apunta a i
            new_auth[i] = sum(hubs[j] for j in inlinks_idx[i])

    # Actualiza hubs a partir de nuevas authorities
    new_hubs = np.zeros(n, dtype=np.float64)
    for i in range(n):
        if outlinks_idx[i]:                       # si i apunta a otros
            new_hubs[i] = sum(new_auth[k] for k in outlinks_idx[i])

    new_auth = _norm2(new_auth)
    new_hubs = _norm2(new_hubs)

    # Criterio de convergencia, norma l1
    delta = np.abs(new_auth - auth).sum() + np.abs(new_hubs - hubs).sum()
    auth, hubs = new_auth, new_hubs
    if delta < tol:
        break

top_h = np.argsort(-hubs)[:10]
top_a = np.argsort(-auth)[:10]
print("Hubs (top 10):", [(int(i), float(hubs[i])) for i in top_h])
print("Authorities (top 10):", [(int(i), float(auth[i])) for i in top_a])


Hubs (top 10): [(3, 0.9999999999983535), (0, 1.8146682853750637e-06), (1, 4.2762695805016335e-13), (2, 6.74676378448719e-43), (13, 6.74676378448719e-43), (5, 2.489206111140362e-60), (15, 2.489206111140362e-60), (6, 2.489206111140362e-60), (7, 3.927274772231721e-90), (4, 3.927274772231721e-90)]
Authorities (top 10): [(3, 0.35355339059260843), (6, 0.35355339059260843), (10, 0.35355339059260843), (17, 0.35355339059260843), (24, 0.35355339059260843), (40, 0.35355339059260843), (47, 0.35355339059260843), (41, 0.35355339059260843), (16, 7.332367143957547e-07), (29, 7.332367143957547e-07)]


**Luego de ejecutar el HITS se almacenan los 10 hubs y auth en en el data frame**



In [ ]:
# Almacenamos los valores en el dataframe
df['hits_hub']  = hubs
df['hits_auth'] = auth
df.head()

,url,text,urls_out,wordrank,pr_085,pr_060,hits_hub,hits_auth
0,https://terraria.fandom.com/es/wiki/Armadura_d...,armadura sombra masculino femenino nombres ing...,"[https://terraria.fandom.com/es/wiki/Rareza, h...","{'sombra': 0.39472556736096837, 'armadura': 0....",0.010894,0.015736,1.814668e-06,7.332367e-07
1,https://terraria.fandom.com/es/wiki/Expriminator,mineral tungsteno caracteristica tipo bloque s...,[https://terraria.fandom.com/es/wiki/Decoracio...,"{'tungsteno': 0.582740486110008, 'mineral': 0....",0.011150,0.015985,4.276270e-13,2.015853e-13
2,https://terraria.fandom.com/es/wiki/Mineral_de...,arma magica tipo arma terrario principal carac...,[https://terraria.fandom.com/es/wiki/Lingote_d...,"{'ninguno': 0.46449852845207407, 'normal': 0.2...",0.011150,0.015985,6.746764e-43,2.015853e-13
3,https://terraria.fandom.com/es/wiki/Armas_m%C3...,lingote estano nombres ingles tin bar caracter...,[https://terraria.fandom.com/es/wiki/Man%C3%A1...,"{'estano': 0.5095652926316093, 'lingote': 0.40...",0.010709,0.015553,1.000000e+00,3.535534e-01
4,https://terraria.fandom.com/es/wiki/Lingote_de...,inframundo infierno capa mas profundo mundo co...,[https://terraria.fandom.com/es/wiki/Inframund...,"{'inframundo': 0.3051795970080333, 'ahora': 0....",0.012730,0.017584,3.927275e-90,6.360910e-43


# Lógica de búsqueda

## Funciones de búsqueda

**Limpieza y normalización de texto**

Limpia el texto: minúsculas, eliminación de signos y stopwords, y aplica lematización o stemming para normalizar palabras antes de vectorizar.

In [ ]:
# Procesa la consulta y genera unigramas + bigramas, para asi matchear contra el wordrank que incluye n gramas (1,2)
def _query_terms(sentence):
  q = text_treatment(sentence)
  toks = nltk.word_tokenize(q)
  bigrams = [f"{toks[i]} {toks[i+1]}" for i in range(len(toks)-1)]
  return toks + bigrams

**Relavancia vs estructura**
El siguiente código define una función que mezcla la relevancia del texto con la importancia estructural de las páginas (su puntaje de PageRank, HITS o SALSA) y lo hace definiendo esta funcion search_with_struct.
Esta función busca páginas relevantes combinando dos tipos de información:
Texto (TF-IDF) = qué tan bien aparece la palabra o frase buscada en el contenido.

Estructura (PageRank, HITS o SALSA) = qué tan importante es la página en el grafo del sitio.

El parámetro alpha ajusta el peso de cada uno:
- Si alpha = 1, solo usa la estructura (como PageRank puro).
- Si alpha = 0, solo usa el texto (búsqueda por contenido).
- Si alpha = 0.5, mezcla ambos a partes iguales..

In [ ]:
def _search_with_struct(sentence, df, struct_col, alpha=0.5):
  q_terms = _query_terms(sentence)
  results = {}
  for _, row in df.iterrows():
    wr = row['wordrank'] or {}
    p_wordrank = 0.0
    for t in q_terms:
      if t in wr:
        p_wordrank += wr[t]
    if p_wordrank != 0:
      struct = row.get(struct_col, None)
      if struct is None:
        continue
      score = alpha * float(struct) + (1 - alpha) * float(p_wordrank)
      results[row['url']] = score

  #Si no hay match semántico, entonces un fallback estructural
  if not results:
    tmp = df[['url', struct_col]].dropna()
    tmp = tmp.sort_values(struct_col, ascending=False).head(10)
    return [(u, float(s)) for u, s in zip(tmp['url'], tmp[struct_col])]

  sorted_list = sorted(results.items(), key=lambda x: x[1], reverse=True)
  return sorted_list[:10]


**SALSA**

Aplica el algoritmo SALSA sobre el grafo para obtener un ranking basado en caminatas aleatorias bipartitas. Combina propiedades similares a HITS y PageRank.

In [ ]:
# SALSA implementation
def salsa(inlinks, outlinks, tol=1e-6, max_iter=200):
    n = len(inlinks)
    # Inicializar hub y authority scores uniformemente
    hubs = np.full(n, 1.0 / n, dtype=np.float64)
    auth = np.full(n, 1.0 / n, dtype=np.float64)

    # normalizacion (L1 norm)
    def _normalize_l1(v):
        s = np.sum(np.abs(v))
        return v / s if s > 0 else v

    for _ in range(max_iter):
        # Actualiza authorities (flujo de hubs a authorities)
        new_auth = np.zeros(n, dtype=np.float64)
        for i in range(n):
            if inlinks[i]:
                # Suma de hubs de nodos que apuntan a i, normalizado por su outdegree
                new_auth[i] = sum(hubs[j] / len(outlinks[j]) for j in inlinks[i] if outlinks[j])

        # Actualiza hubs (flujo de authorities a hubs)
        new_hubs = np.zeros(n, dtype=np.float64)
        for i in range(n):
            if outlinks[i]:
                 # Suma de authorities de nodos a los que i apunta, normalizado por su indegree
                new_hubs[i] = sum(auth[k] / len(inlinks[k]) for k in outlinks[i] if inlinks[k])

        # Normalizar scores
        new_auth = _normalize_l1(new_auth)
        new_hubs = _normalize_l1(new_hubs)

        # Criterio de convergencia (L1 norm of the change)
        delta = np.sum(np.abs(new_auth - auth)) + np.sum(np.abs(new_hubs - hubs))
        auth, hubs = new_auth, new_hubs
        if delta < tol:
            break

    return hubs, auth

# Re-calculamos inlinks y outlinks para SALSA
n = len(df)
outlinks_salsa = [np.where(pagerank[i, :] != 0)[0].tolist() for i in range(n)]
inlinks_salsa  = [np.where(pagerank[:, j] != 0)[0].tolist() for j in range(n)]


# Calcular SALSA scores
salsa_hubs, salsa_auth = salsa(inlinks_salsa, outlinks_salsa, tol=1e-6, max_iter=200)

# Almacenar los valores en el dataframe
df['salsa_hub']  = salsa_hubs
df['salsa_auth'] = salsa_auth

display(df.head())

,url,text,urls_out,wordrank,pr_085,pr_060,hits_hub,hits_auth,salsa_hub,salsa_auth
0,https://terraria.fandom.com/es/wiki/Armadura_d...,armadura sombra masculino femenino nombres ing...,"[https://terraria.fandom.com/es/wiki/Rareza, h...","{'sombra': 0.39472556736096837, 'armadura': 0....",0.010894,0.015736,1.814668e-06,7.332367e-07,0.05,0.022222
1,https://terraria.fandom.com/es/wiki/Expriminator,mineral tungsteno caracteristica tipo bloque s...,[https://terraria.fandom.com/es/wiki/Decoracio...,"{'tungsteno': 0.582740486110008, 'mineral': 0....",0.011150,0.015985,4.276270e-13,2.015853e-13,0.05,0.022222
2,https://terraria.fandom.com/es/wiki/Mineral_de...,arma magica tipo arma terrario principal carac...,[https://terraria.fandom.com/es/wiki/Lingote_d...,"{'ninguno': 0.46449852845207407, 'normal': 0.2...",0.011150,0.015985,6.746764e-43,2.015853e-13,0.05,0.022222
3,https://terraria.fandom.com/es/wiki/Armas_m%C3...,lingote estano nombres ingles tin bar caracter...,[https://terraria.fandom.com/es/wiki/Man%C3%A1...,"{'estano': 0.5095652926316093, 'lingote': 0.40...",0.010709,0.015553,1.000000e+00,3.535534e-01,0.05,0.022222
4,https://terraria.fandom.com/es/wiki/Lingote_de...,inframundo infierno capa mas profundo mundo co...,[https://terraria.fandom.com/es/wiki/Inframund...,"{'inframundo': 0.3051795970080333, 'ahora': 0....",0.012730,0.017584,3.927275e-90,6.360910e-43,0.05,0.022222




# Funciones de búsqueda: PageRank, HITS y SALSA

Este bloque define cuatro funciones de búsqueda, cada una basada en un tipo distinto de ranking estructural (PageRank, HITS o SALSA).
Todas utilizan la misma función auxiliar _search_with_struct, que combina contenido textual (wordrank) y estructura para obtener las páginas más relevantes.

- Pagerank: Se usa la columna 'pr_085', que contiene los valores de PageRank calculados con un damping factor = 0.85. El parámetro alpha = 0.5 indica que el puntaje final mezcla 50% de relevancia estructural (PageRank) y 50% de coincidencia textual (wordrank).
El resultado son las 10 páginas más relevantes combinando ambos criterios
tambien se uso PageRank con un damping factor diferente (0.60), esto sirve para comparar cómo cambia el orden de las páginas cuando se da menos peso a los enlaces y más al salto aleatorio. Permite analizar la sensibilidad del algoritmo frente al parámetro d.

- HITS: Esta función usa la columna 'hits_auth', que contiene los valores de authorities calculados por el algoritmo HITS.
Las páginas con alto valor en hits_auth son consideradas fuentes confiables (porque las enlazan buenos hubs).
El resultado pondera mitad estructura (importancia como authority) y mitad texto (coincidencia con la búsqueda).

- SALSA: Aquí se usan los valores de autoridad de SALSA, un método que combina ideas de PageRank y HITS.
SALSA busca equilibrar el peso de hubs y authorities, produciendo rankings más estables, el cálculo también pondera mitad estructura y mitad contenido.

In [ ]:
# Pagerank con d=0.85
# Busqueda usando pagerank con d=0.85 y wordrank
def search_pagerank(sentence, df):
  return _search_with_struct(sentence, df, struct_col='pr_085', alpha=0.5)

# Pagerank con d 0.60
def search_pagerank_alt(sentence, df):
  col = 'pr_060'
  return _search_with_struct(sentence, df, struct_col=col, alpha=0.5)

# Usando authorities como señal principal
def search_hits(sentence, df):
  return _search_with_struct(sentence, df, struct_col='hits_auth', alpha=0.5)

#SALSA
def search_salsa(sentence, df):
  return _search_with_struct(sentence, df, struct_col='salsa_auth', alpha=0.5)


## Consultas
A continuacion hacesmos las consultas y probamos cada algoritmo para asi poder concluir.

In [ ]:
import nltk
nltk.download('punkt_tab')


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

**Resultados para busqueda de palabras sueltas**

In [ ]:
# Análisis: "Jefes" es un término central, probablemente en páginas con alta autoridad.
print("Consulta: jefes (usando SALSA)")
top_terraria_jefes = search_salsa("jefes", df)
for i, (url, score) in enumerate(top_terraria_jefes, start=1):
  print(f"{i:>2}) {score:.4f}  {url}")
print("-" * 30)

# Análisis: "Armas" es un tema amplio, páginas con listas de armas podrían tener buena autoridad.
print("Consulta: armas (usando HITS)")
top_terraria_armas = search_hits("armas", df)
for i, (url, score) in enumerate(top_terraria_armas, start=1):
  print(f"{i:>2}) {score:.4f}  {url}")
print("-" * 30)

# Análisis: "Biomas" son ubicaciones clave, páginas relevantes podrían tener alta autoridad.
print("Consulta: biomas (usando PageRank 0.85)")
top_terraria_biomas = search_pagerank("biomas", df)
for i, (url, score) in enumerate(top_terraria_biomas, start=1):
  print(f"{i:>2}) {score:.4f}  {url}")
print("-" * 30)

# Análisis: "Objetos" es un término muy general. PageRank podría ayudar a encontrar páginas centrales sobre objetos.
print("Consulta: objetos (usando PageRank 0.60)")
top_terraria_objetos = search_pagerank_alt("objetos", df)
for i, (url, score) in enumerate(top_terraria_objetos, start=1):
  print(f"{i:>2}) {score:.4f}  {url}")
print("-" * 30)

# Análisis: "Minerales" son ítems de crafteo, probablemente en páginas con listas o guías.
print("Consulta: minerales (usando SALSA)")
top_terraria_minerales = search_salsa("minerales", df)
for i, (url, score) in enumerate(top_terraria_minerales, start=1):
  print(f"{i:>2}) {score:.4f}  {url}")
print("-" * 30)

Consulta: jefes (usando SALSA)
 1) 0.0719  https://terraria.fandom.com/es/wiki/Lingote_de_esta%C3%B1o
 2) 0.0506  https://terraria.fandom.com/es/wiki/Trufa
 3) 0.0499  https://terraria.fandom.com/es/wiki/El_novio
 4) 0.0437  https://terraria.fandom.com/es/wiki/Cajas_de_pesca
 5) 0.0407  https://terraria.fandom.com/es/wiki/Forja
------------------------------
Consulta: armas (usando HITS)
 1) 0.2328  https://terraria.fandom.com/es/wiki/Espina_vil
 2) 0.2157  https://terraria.fandom.com/es/wiki/Tridente
 3) 0.1181  https://terraria.fandom.com/es/wiki/Bolsas_del_tesoro
 4) 0.0777  https://terraria.fandom.com/es/wiki/Mineral_de_tungsteno
 5) 0.0572  https://terraria.fandom.com/es/wiki/Defensa
 6) 0.0525  https://terraria.fandom.com/es/wiki/Duende_guerrero
 7) 0.0505  https://terraria.fandom.com/es/wiki/Superficie
 8) 0.0388  https://terraria.fandom.com/es/wiki/Mecanismos
 9) 0.0322  https://terraria.fandom.com/es/wiki/Inframundo
------------------------------
Consulta: biomas (usando PageR

**Resultados para consultas de oraciones (usando el método de búsqueda más apropiado):**

In [32]:
# Consulta: como invocar al ojo de cthulhu
# Análisis: Consulta muy específica. Relevancia textual es clave. SALSA Authority podría complementar.
print("Consulta: como invocar al ojo de cthulhu (usando SALSA)")
top_terraria_cthulhu = search_salsa("como invocar al ojo de cthulhu", df)
for i, (url, score) in enumerate(top_terraria_cthulhu, start=1):
  print(f"{i:>2}) {score:.4f}  {url}")
print("-" * 30)

# Consulta: que hacer despues de derrotar al muro carnoso
# Análisis: Relacionado con progresión y jefes. HITS Authority podría ser útil para encontrar páginas que enlacen a contenido relevante.
print("Consulta: que hacer despues de derrotar al muro carnoso (usando HITS)")
top_terraria_postmuro = search_hits("que hacer despues de derrotar al muro carnoso", df)
for i, (url, score) in enumerate(top_terraria_postmuro, start=1):
  print(f"{i:>2}) {score:.4f}  {url}")
print("-" * 30)

# Consulta: como conseguir la llave de la jungla
# Análisis: Específico sobre un objeto. SALSA Authority para páginas sobre objetos o biomas.
print("Consulta: como conseguir la llave de la jungla (usando SALSA)")
top_terraria_llave = search_salsa("como conseguir la llave de la jungla", df)
for i, (url, score) in enumerate(top_terraria_llave, start=1):
  print(f"{i:>2}) {score:.4f}  {url}")
print("-" * 30)

# Consulta: cual es la mejor armadura pre duro
# Análisis: Específico sobre crafteo/equipo. HITS Authority para páginas que enlacen a muchas armaduras o guías.
print("Consulta: cual es la mejor armadura pre duro (usando HITS)")
top_terraria_armadurapre = search_hits("cual es la mejor armadura pre duro", df)
for i, (url, score) in enumerate(top_terraria_armadurapre, start=1):
  print(f"{i:>2}) {score:.4f}  {url}")
print("-" * 30)

# Consulta: donde encontrar el altar demoniaco carmesí
# Análisis: Específico sobre ubicación/estructura. PageRank podría ser útil para páginas centrales sobre biomas o altares.
print("Consulta: donde encontrar el altar demoniaco carmesí (usando PageRank 0.85)")
top_terraria_altar = search_pagerank("donde encontrar el altar demoniaco carmesí", df)
for i, (url, score) in enumerate(top_terraria_altar, start=1):
  print(f"{i:>2}) {score:.4f}  {url}")
print("-" * 30)

Consulta: como invocar al ojo de cthulhu (usando SALSA)
 1) 0.0701  https://terraria.fandom.com/es/wiki/El_novio
 2) 0.0448  https://terraria.fandom.com/es/wiki/Forja
 3) 0.0410  https://terraria.fandom.com/es/wiki/Mineral_de_oro
 4) 0.0385  https://terraria.fandom.com/es/wiki/Champi%C3%B1%C3%B3n_feroz
 5) 0.0351  https://terraria.fandom.com/es/wiki/Tiempo
------------------------------
Consulta: que hacer despues de derrotar al muro carnoso (usando HITS)
 1) 0.3413  https://terraria.fandom.com/es/wiki/Lingote_de_esta%C3%B1o
 2) 0.2383  https://terraria.fandom.com/es/wiki/Arco_de_madera
 3) 0.2001  https://terraria.fandom.com/es/wiki/Tridente
 4) 0.0454  https://terraria.fandom.com/es/wiki/Forja
 5) 0.0452  https://terraria.fandom.com/es/wiki/Expriminator
 6) 0.0373  https://terraria.fandom.com/es/wiki/Terraria
 7) 0.0232  https://terraria.fandom.com/es/wiki/Mineral_de_hierro
------------------------------
Consulta: como conseguir la llave de la jungla (usando SALSA)
 1) 0.1131  https:

# Conclusiones

Los resultados obtenidos en las diferentes consultas reflejan el comportamiento esperado de cada algoritmo y la forma en que la relevancia estructural y la relevancia semántica interactúan al realizar las búsquedas en el wiki de Terraria.

En las consultas más simples como “jefes”, “armas”, “biomas”, “objetos” y “minerales”, se observa que los algoritmos estructurales priorizan distintos tipos de nodos dentro del grafo:

HITS (basado en authorities) ofrece resultados más temáticos, destacando artículos estrechamente interconectados, como Espina vil o Tridente en la búsqueda de “armas”.

PageRank (d = 0.85) resalta nodos con alta conectividad global, como Forja o Mazmorra, que aparecen frecuentemente enlazados en muchas secciones.

PageRank (d = 0.60) genera un ranking más equilibrado, con páginas más diversas al reducir la dependencia estructural del damping factor.

SALSA, en cambio, ofrece una mezcla entre coherencia temática y variedad, incluyendo páginas con buena conectividad local pero también con peso textual, como Meteorito y Mineral de oro en la búsqueda de “minerales”.

En las consultas más complejas, que incluyen oraciones completas, los patrones se mantienen pero se hacen más evidentes los límites de cada algoritmo.
Por ejemplo, en “cómo invocar al ojo de Cthulhu”, el método SALSA arrojó resultados como Meteorito, Luna o Corrupción, que si bien no son el artículo exacto, pertenecen al conjunto de páginas fuertemente relacionadas con eventos o condiciones previas al enfrentamiento de jefes. Esto demuestra que SALSA logra captar contexto estructural incluso cuando el texto exacto no coincide.

En la búsqueda “qué hacer después de derrotar al muro carnoso”, el algoritmo HITS priorizó páginas como Taller del Reparador, Meteorito y Objetos de apariencia. Estas páginas suelen ser destinos comunes de enlaces desde guías o tutoriales del modo hardmode, lo que indica que HITS es especialmente sensible a subredes temáticas densas, donde muchas páginas se enlazan entre sí sobre una misma fase del juego.

Por otro lado, en “cómo conseguir la llave de la jungla”, los resultados de SALSA incluyeron páginas como Reciclar, Cueva de champiñones y Generador de Obsidiana. Si bien no son respuestas directas, reflejan que el algoritmo prioriza nodos con conectividad local alta, es decir, páginas visitadas desde contenidos cercanos en estructura aunque no sean textualmente equivalentes.

Finalmente, la búsqueda “dónde encontrar el altar demoníaco carmesí”, realizada con PageRank (d = 0.85), mostró una fuerte concentración en páginas centrales del wiki como Subsuelo, IDs de objetos y Desierto. Esto confirma que PageRank tiende a favorecer nodos muy enlazados a nivel global, incluso si no están semánticamente relacionados con la consulta, pero que funcionan como puntos de acceso importantes dentro de la red.

En conjunto, los resultados permiten concluir que:

HITS es el método más coherente en temas bien interconectados o localizados (por ejemplo, “armas” o “qué hacer después del muro carnoso”).

SALSA logra un balance entre estructura y contenido, siendo útil para consultas más contextuales o narrativas (“cómo invocar al ojo de Cthulhu”).

PageRank (0.85) resalta la popularidad global de las páginas y es más estable, pero menos preciso semánticamente.

PageRank (0.60) introduce mayor diversidad al ranking, sacrificando algo de consistencia estructural.

En definitiva, el modelo implementado demuestra que la combinación de TF-IDF con métricas estructurales (PageRank, HITS y SALSA) permite construir un buscador que no solo considera el texto, sino también la posición de cada página dentro del ecosistema del sitio, ofreciendo una visión más completa de la relevancia en entornos de información interconectados como los wikis